In [1]:
from pyscf import gto, dft
from pyscf.sftda.uhf_sf import get_ab_sf
import numpy as np
from pyscf.data.nist import HARTREE2EV

atom = '''
  6  -0.664855   -0.020012    0.000000
  1  -1.083681    0.985074   -0.000001
  1  -1.030336   -0.543074   -0.890185
  1  -1.030336   -0.543072    0.890186
  8   0.748866    0.121952    0.000000
  1   1.142557   -0.754472    0.000000
'''
mol = gto.M(atom=atom, charge=0, spin=0, basis='cc-pvtz')

In [2]:
fun_list = ['HF', 'SVWN', 'BLYP', 'PBE', 'TPSS', 'B3LYP', 'PBE0', 'M062X', 'SCAN0']
for fun in fun_list:
    print('----------')
    print(fun)
    mf0 = mol.RKS(xc=fun)
    mf0.kernel()

    print('Spin-conserving: ')
    td = mf0.TDDFT()
    td.singlet = False
    sc = td.kernel(nstates=5)[0] * HARTREE2EV
    
    mf = mol.UKS(xc=fun)
    mf.mo_energy = np.array([mf0.mo_energy, mf0.mo_energy])
    mf.mo_coeff = np.array([mf0.mo_coeff, mf0.mo_coeff])
    mf.mo_occ = np.array([mf0.mo_occ*0.5, mf0.mo_occ*0.5])

    a, b = get_ab_sf(mf)
    A_baba, A_abab = a
    B_baab, B_abba = b

    n_occ_a, n_virt_b = A_abab.shape[0], A_abab.shape[1]
    n_occ_b, n_virt_a = B_abba.shape[2], B_abba.shape[3]

    A_abab_2d = A_abab.reshape((n_occ_a*n_virt_b,n_occ_a*n_virt_b), order='C')
    B_abba_2d = B_abba.reshape((n_occ_a*n_virt_b,n_occ_b*n_virt_a), order='C')
    B_baab_2d = B_baab.reshape((n_occ_b*n_virt_a,n_occ_a*n_virt_b), order='C')
    A_baba_2d = A_baba.reshape((n_occ_b*n_virt_a,n_occ_b*n_virt_a), order='C')

    Casida_matrix = np.block([[ A_abab_2d, B_abba_2d],
                            [-B_baab_2d,-A_baba_2d]])

    eigenvals, eigenvecs = np.linalg.eig(Casida_matrix)

    idxt = eigenvals.real.argsort()
    eigenvals = eigenvals[idxt]
    eigenvecs = eigenvecs[:, idxt]

    norms = np.linalg.norm(eigenvecs[:n_occ_a * n_virt_b], axis=0)**2
    norms -= np.linalg.norm(eigenvecs[n_occ_a * n_virt_b:], axis=0)**2

    sfd_eigenvals = eigenvals[norms>0]
    sfd_eigenvecs = eigenvecs[:, norms>0].transpose(1,0)

    print('Spin-flip down: ')
    sfd = sfd_eigenvals * HARTREE2EV
    print(sfd[:5])


    print('Spin-flip up: ')
    sfu_eigenvals = eigenvals[norms<0]
    sfu = - sfu_eigenvals[::-1] * HARTREE2EV
    print(sfu[:5])

    np.save('sfd_'+fun, sfd)
    np.save('sfu_'+fun, sfu)
    np.save('sc_'+fun, sc)

----------
HF
converged SCF energy = -115.089123392699
Spin-conserving: 
Excited State energies (eV)
[ 7.88889242  9.4175311   9.48267833 10.13133208 10.47726273]
200
Spin-flip down: 
[ 7.88889242  9.4175311   9.48267833 10.13133208 10.47726273]
Spin-flip up: 
[ 7.88889242  9.4175311   9.48267833 10.13133208 10.47726273]
----------
SVWN
converged SCF energy = -114.834343832455
Spin-conserving: 
Excited State energies (eV)
[6.23277405 7.3174398  7.89183756 8.46297054 8.54822451]
200
Spin-flip down: 
[6.23277405 7.3174398  7.89183756 8.46297054 8.54822451]
Spin-flip up: 
[6.23277405 7.3174398  7.89183756 8.46297054 8.54822451]
----------
BLYP
converged SCF energy = -115.728292769164
Spin-conserving: 
Excited State energies (eV)
[6.01740925 7.05382458 7.69971687 8.26034947 8.36399406]
200
Spin-flip down: 
[6.01740925 7.05382458 7.69971687 8.26034947 8.36399406]
Spin-flip up: 
[6.01740925 7.05382458 7.69971687 8.26034947 8.36399406]
----------
PBE
converged SCF energy = -115.625407529089
S

/home/peter/anaconda3/envs/sftdgrad/lib/python3.9/site-packages/pyscf/dft/libxc.py:512: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, corresponding to the original definition by Stephens et al. (issue 1480) and the same as the B3LYP functional in Gaussian. To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '


converged SCF energy = -115.772248235449
Spin-conserving: 
Excited State energies (eV)
[6.54960687 7.81296413 8.2562082  9.02704671 9.16445416]
200
Spin-flip down: 
[6.54960687 7.81296413 8.2562082  9.02704671 9.16445416]
Spin-flip up: 
[6.54960687 7.81296413 8.2562082  9.02704671 9.16445416]
----------
PBE0
converged SCF energy = -115.633543212404
Spin-conserving: 
Excited State energies (eV)
[6.78990953 8.06657539 8.49304724 9.22447879 9.42585954]
200
Spin-flip down: 
[6.78990953 8.06657539 8.49304724 9.22447879 9.42585954]
Spin-flip up: 
[6.78990953 8.06657539 8.49304724 9.22447879 9.42585954]
----------
M062X
converged SCF energy = -115.714063220107
Spin-conserving: 
Excited State energies (eV)
[ 7.01457801  8.78806447  8.8238341   9.81996136 10.06154215]
200
Spin-flip down: 
[ 7.01457801  8.78806447  8.8238341   9.81996136 10.06154215]
Spin-flip up: 
[ 7.01457801  8.78806447  8.8238341   9.81996136 10.06154215]
----------
SCAN0
converged SCF energy = -115.697985630394
Spin-conserv